# 02 — Validación, Limpieza y Estandarización de Datos

Este notebook transforma el dataset bruto de permisos de circulación en un dataset limpio y confiable.

**Pipeline de limpieza:**
1. Perfilado inicial del dataset (dimensiones, tipos, distribuciones)
2. Detección de problemas de calidad
3. Limpieza y estandarización
4. Validación final y reporte de calidad

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from config.config import RAW_FILE, CLEAN_FILE, DATA_PROCESSED_DIR, TIPOS_VEHICULO, ZONAS
import os

## 1. Carga y perfilado inicial

In [2]:
df = pd.read_csv(RAW_FILE)

print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nColumnas y tipos:")
print(df.dtypes)
print(f"\nPrimeras 5 filas:")
df.head()

Dimensiones: 5050 filas x 9 columnas

Columnas y tipos:
id_permiso                int64
tipo_vehiculo            object
fecha_emision            object
duracion_dias           float64
zona_circulacion         object
monto_pagado            float64
renovacion                 bool
infracciones_previas      int64
estado                   object
dtype: object

Primeras 5 filas:


,id_permiso,tipo_vehiculo,fecha_emision,duracion_dias,zona_circulacion,monto_pagado,renovacion,infracciones_previas,estado
0,1,Furgoneta,2025-05-24 00:00:00,28.0,Zona D,102875.69,False,1,Inactivo
1,2,Bicicleta,2025-05-10 00:00:00,20.0,Zona C,6341.74,False,0,Activo
2,3,Camion,2025-02-08 00:00:00,5.0,Zona A,92238.12,False,1,Inactivo
3,4,Bicicleta,2024-11-22 00:00:00,8.0,Zona D,9604.68,False,1,Inactivo
4,5,Bicicleta,2024-11-03 00:00:00,12.0,Zona D,9292.32,False,2,Activo


In [3]:
# Reporte de calidad inicial
quality_report = pd.DataFrame({
    'tipo': df.dtypes,
    'no_nulos': df.notnull().sum(),
    'nulos': df.isnull().sum(),
    'pct_nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'unicos': df.nunique(),
})
print("=== REPORTE DE CALIDAD INICIAL ===")
quality_report

=== REPORTE DE CALIDAD INICIAL ===


,tipo,no_nulos,nulos,pct_nulos,unicos
id_permiso,int64,5050,0,0.00,5000
tipo_vehiculo,object,5050,0,0.00,14
fecha_emision,object,5050,0,0.00,365
duracion_dias,float64,4803,247,4.89,30
zona_circulacion,object,4946,104,2.06,4
monto_pagado,float64,5050,0,0.00,4941
renovacion,bool,5050,0,0.00,2
infracciones_previas,int64,5050,0,0.00,6
estado,object,5050,0,0.00,2


## 2. Detección de problemas de calidad

Identificamos cada tipo de problema antes de limpiar.

In [4]:
# 2.1 Duplicados
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {n_duplicados} ({n_duplicados/len(df)*100:.2f}%)")

# 2.2 Valores faltantes
print(f"\nValores faltantes por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2.3 Inconsistencias en tipo_vehiculo
print(f"\nValores únicos en tipo_vehiculo ({df['tipo_vehiculo'].nunique()}):")
print(df['tipo_vehiculo'].value_counts())

Filas duplicadas: 50 (0.99%)

Valores faltantes por columna:
duracion_dias       247
zona_circulacion    104
dtype: int64

Valores únicos en tipo_vehiculo (14):
tipo_vehiculo
Coche        827
Furgoneta    801
Bicicleta    794
Monopatin    788
Moto         771
Camion       770
coche        100
OTRO          51
BICICLETA     29
CAMION        28
MONOPATIN     26
COCHE         24
MOTO          21
FURGONETA     20
Name: count, dtype: int64


In [5]:
# 2.4 Valores fuera de rango
n_duracion_neg = (df['duracion_dias'] < 0).sum()
print(f"duracion_dias negativos: {n_duracion_neg}")

n_monto_neg = (df['monto_pagado'] < 0).sum()
print(f"monto_pagado negativos: {n_monto_neg}")

# 2.5 Fechas inválidas
fechas_parsed = pd.to_datetime(df['fecha_emision'], errors='coerce')
n_fechas_invalidas = fechas_parsed.isnull().sum() - df['fecha_emision'].isnull().sum()
print(f"Fechas con formato inválido: {n_fechas_invalidas}")

# 2.6 Categorías no esperadas
categorias_validas = [t.capitalize() for t in TIPOS_VEHICULO]
df_tipo_norm = df['tipo_vehiculo'].str.strip().str.capitalize()
categorias_raras = df_tipo_norm[~df_tipo_norm.isin(categorias_validas) & df_tipo_norm.notna()]
print(f"\nCategorías no esperadas en tipo_vehiculo: {categorias_raras.nunique()}")
if len(categorias_raras) > 0:
    print(categorias_raras.value_counts())

duracion_dias negativos: 100
monto_pagado negativos: 51
Fechas con formato inválido: 50

Categorías no esperadas en tipo_vehiculo: 1
tipo_vehiculo
Otro    51
Name: count, dtype: int64


## 3. Limpieza y estandarización

Aplicamos correcciones en orden lógico: duplicados → formato → nulos → outliers → fechas.

In [6]:
n_antes = len(df)

# 3.1 Eliminar duplicados
df.drop_duplicates(inplace=True)
print(f"[Duplicados] {n_antes} → {len(df)} filas (eliminados: {n_antes - len(df)})")

# 3.2 Estandarizar tipo_vehiculo
df['tipo_vehiculo'] = df['tipo_vehiculo'].str.strip().str.capitalize()
# Mapear categoría 'Otro' a NaN para imputar después
df.loc[df['tipo_vehiculo'] == 'Otro', 'tipo_vehiculo'] = np.nan
print(f"[Formato] tipo_vehiculo estandarizado. Valores únicos: {df['tipo_vehiculo'].nunique()}")

# 3.3 Parsear fechas (las inválidas se convierten en NaT)
df['fecha_emision'] = pd.to_datetime(df['fecha_emision'], errors='coerce')
n_fechas_nat = df['fecha_emision'].isnull().sum()
print(f"[Fechas] Fechas inválidas convertidas a NaT: {n_fechas_nat}")

# Eliminar filas con fecha inválida (no se puede imputar una fecha razonablemente)
df.dropna(subset=['fecha_emision'], inplace=True)
print(f"[Fechas] Filas restantes tras eliminar fechas NaT: {len(df)}")

[Duplicados] 5050 → 5000 filas (eliminados: 50)
[Formato] tipo_vehiculo estandarizado. Valores únicos: 6
[Fechas] Fechas inválidas convertidas a NaT: 50
[Fechas] Filas restantes tras eliminar fechas NaT: 4950


In [7]:
# 3.4 Corregir valores fuera de rango
# duracion_dias: reemplazar negativos y NaN con la mediana
mediana_duracion = df.loc[df['duracion_dias'] > 0, 'duracion_dias'].median()
n_neg = (df['duracion_dias'] < 0).sum()
n_null = df['duracion_dias'].isnull().sum()
df.loc[df['duracion_dias'] < 0, 'duracion_dias'] = mediana_duracion
df['duracion_dias'].fillna(mediana_duracion, inplace=True)
print(f"[duracion_dias] Negativos corregidos: {n_neg}, Nulos imputados: {n_null} (mediana={mediana_duracion})")

# monto_pagado: reemplazar negativos con la mediana por tipo_vehiculo
n_monto_neg = (df['monto_pagado'] < 0).sum()
medianas_monto = df.loc[df['monto_pagado'] > 0].groupby('tipo_vehiculo')['monto_pagado'].median()
for tipo, mediana in medianas_monto.items():
    mask = (df['monto_pagado'] < 0) & (df['tipo_vehiculo'] == tipo)
    df.loc[mask, 'monto_pagado'] = mediana
# Si quedan negativos sin tipo (por NaN en tipo_vehiculo)
df.loc[df['monto_pagado'] < 0, 'monto_pagado'] = df.loc[df['monto_pagado'] > 0, 'monto_pagado'].median()
print(f"[monto_pagado] Negativos corregidos: {n_monto_neg}")

# 3.5 Imputar nulos categóricos
n_null_zona = df['zona_circulacion'].isnull().sum()
df['zona_circulacion'].fillna(df['zona_circulacion'].mode()[0], inplace=True)
print(f"[zona_circulacion] Nulos imputados con moda: {n_null_zona}")

n_null_tipo = df['tipo_vehiculo'].isnull().sum()
df['tipo_vehiculo'].fillna(df['tipo_vehiculo'].mode()[0], inplace=True)
print(f"[tipo_vehiculo] Nulos imputados con moda: {n_null_tipo}")

[duracion_dias] Negativos corregidos: 100, Nulos imputados: 246 (mediana=15.0)
[monto_pagado] Negativos corregidos: 50
[zona_circulacion] Nulos imputados con moda: 99
[tipo_vehiculo] Nulos imputados con moda: 50


C:\Users\matya\AppData\Local\Temp\ipykernel_14380\2777599096.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['duracion_dias'].fillna(mediana_duracion, inplace=True)
C:\Users\matya\AppData\Local\Temp\ipykernel_14380\2777599096.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a cop

## 4. Validación final y reporte de calidad

In [8]:
# Reporte de calidad post-limpieza
quality_post = pd.DataFrame({
    'tipo': df.dtypes,
    'no_nulos': df.notnull().sum(),
    'nulos': df.isnull().sum(),
    'pct_nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'unicos': df.nunique(),
})

print("=== REPORTE DE CALIDAD POST-LIMPIEZA ===")
print(f"Filas: {len(df)}")
print(f"Duplicados: {df.duplicated().sum()}")
print(f"Nulos totales: {df.isnull().sum().sum()}")
print(f"duracion_dias min: {df['duracion_dias'].min()}, max: {df['duracion_dias'].max()}")
print(f"monto_pagado min: {df['monto_pagado'].min():.2f}")
print(f"\nDistribución tipo_vehiculo:")
print(df['tipo_vehiculo'].value_counts())
print(f"\nDistribución estado:")
print(df['estado'].value_counts(normalize=True).round(3))
quality_post

=== REPORTE DE CALIDAD POST-LIMPIEZA ===
Filas: 4950
Duplicados: 0
Nulos totales: 0
duracion_dias min: 1.0, max: 29.0
monto_pagado min: 1000.00

Distribución tipo_vehiculo:
tipo_vehiculo
Coche        986
Bicicleta    811
Furgoneta    806
Monopatin    793
Camion       779
Moto         775
Name: count, dtype: int64

Distribución estado:
estado
Activo      0.689
Inactivo    0.311
Name: proportion, dtype: float64


,tipo,no_nulos,nulos,pct_nulos,unicos
id_permiso,int64,4950,0,0.0,4950
tipo_vehiculo,object,4950,0,0.0,6
fecha_emision,datetime64[ns],4950,0,0.0,364
duracion_dias,float64,4950,0,0.0,29
zona_circulacion,object,4950,0,0.0,4
monto_pagado,float64,4950,0,0.0,4896
renovacion,bool,4950,0,0.0,2
infracciones_previas,int64,4950,0,0.0,6
estado,object,4950,0,0.0,2


In [9]:
# Validaciones (assertions)
assert df.isnull().sum().sum() == 0, "Quedan valores nulos"
assert df.duplicated().sum() == 0, "Quedan duplicados"
assert (df['duracion_dias'] > 0).all(), "Hay duraciones no positivas"
assert (df['monto_pagado'] > 0).all(), "Hay montos no positivos"
assert df['tipo_vehiculo'].isin(['Coche', 'Moto', 'Camion', 'Furgoneta', 'Bicicleta', 'Monopatin']).all(), "Categorías inválidas en tipo_vehiculo"
assert df['zona_circulacion'].isin(['Zona A', 'Zona B', 'Zona C', 'Zona D']).all(), "Categorías inválidas en zona"

print("Todas las validaciones pasaron correctamente.")

Todas las validaciones pasaron correctamente.


In [10]:
# Guardar dataset limpio
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
df.to_csv(CLEAN_FILE, index=False)
print(f"Dataset limpio guardado en: {CLEAN_FILE}")
print(f"Registros finales: {len(df)}")

Dataset limpio guardado en: C:\Users\matya\Desktop\Mis Proyectos\Permisos de circulacion\data\processed\permisos_circulacion_limpio.csv
Registros finales: 4950
